In [ ]:
import glob
import json
import os
import sys
import time
from collections import deque
from pathlib import Path

import cudf
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import seaborn as sns
import shap
import wandb
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

In [ ]:
class CFG:
    model = "xgb"
    data_id = "026"
    seed = 42
    n_fold = 5
    study_name = f"{model}-{data_id}"
    trial_number = None
    custom_number = None
    params_hash = None

In [ ]:
import hashlib
import json
import time
from pathlib import Path


def param_hash(params: dict, n=12) -> str:
    dump = json.dumps(params, sort_keys=True, separators=(",", ":"))
    return hashlib.blake2b(dump.encode(), digest_size=16).hexdigest()[:n]


def make_run_id(
    model, data_id, seed, n_fold, param_src=None, trial=None, custom=None, params=None
):
    cvname = f"cv{n_fold}"
    parts = [model, f"{data_id}", f"s{seed}", cvname]
    suffix = []
    if trial is not None:
        suffix.append(f"tr{trial}")
    if custom:
        suffix.append(f"cus{custom}")
    if params is not None:
        suffix.append(f"h{param_hash(params)}")
    if suffix:
        parts.append("-".join(suffix))
    return "-".join(parts)

# xgb-026-s42-cv5-tr35-h3fa91c2e
# 例
run_id = make_run_id("xgb", "026", 42, 5, trial=35, params=self.params)
run_dir = Path("runs") / run_id
run_dir.mkdir(parents=True, exist_ok=True)

# manifest 保存
manifest = {
    "run_id": run_id,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model": "xgb",
    "data_id": "026",
    "seed": 42,
    "n_fold": 5,
    "study_name": "xgb-026",
    "trial_number": 35,
    "params": self.params,
    "params_hash": param_hash(self.params),
}
(Path(run_dir) / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False)
)